# Week 05 Lab — Simple Neural Networks and Neural Language Models

## Learning goals
- Implement **activation functions** and their derivatives.
- Build a **feedforward neural network** (forward pass, backprop, SGD) from scratch with NumPy.
- Implement a **fixed-window neural language model** with learned word embeddings.
- Evaluate **perplexity** and compare with n-gram baselines.
- Inspect **learned embeddings** via cosine nearest neighbors.

## Part 0 — Setup

In [ ]:
!pip install -q numpy nltk matplotlib

In [ ]:
import math
import random
from collections import Counter, defaultdict

import numpy as np
import matplotlib.pyplot as plt
import nltk

nltk.download("punkt", quiet=True)
random.seed(42)
np.random.seed(42)

---
## Part 1 — Activation Functions and Their Derivatives

Activation functions introduce non-linearity. You need both the function and its derivative for backpropagation.

In [ ]:
def sigmoid(z: np.ndarray) -> np.ndarray:
    """Element-wise sigmoid: 1 / (1 + exp(-z))"""
    # TODO: implement
    pass

def sigmoid_grad(z: np.ndarray) -> np.ndarray:
    """Derivative of sigmoid w.r.t. z: sigma(z) * (1 - sigma(z))"""
    # TODO: implement
    pass

def tanh_act(z: np.ndarray) -> np.ndarray:
    """Element-wise tanh."""
    # TODO: implement (hint: np.tanh)
    pass

def tanh_grad(z: np.ndarray) -> np.ndarray:
    """Derivative of tanh w.r.t. z: 1 - tanh(z)^2"""
    # TODO: implement
    pass

def relu(z: np.ndarray) -> np.ndarray:
    """Element-wise ReLU: max(0, z)"""
    # TODO: implement
    pass

def relu_grad(z: np.ndarray) -> np.ndarray:
    """Derivative of ReLU w.r.t. z: 1 if z > 0 else 0"""
    # TODO: implement
    pass

def softmax(z: np.ndarray) -> np.ndarray:
    """Numerically stable softmax over 1-D vector z."""
    # TODO: implement (subtract max before exponentiating)
    pass

# Quick sanity checks
z = np.array([-2.0, -1.0, 0.0, 1.0, 2.0])
print("sigmoid:", sigmoid(z))
print("tanh   :", tanh_act(z))
print("relu   :", relu(z))
print("softmax:", softmax(z), "sum =", softmax(z).sum())

In [ ]:
# Coding Task B — Visualize activations and their derivatives
xs = np.linspace(-4, 4, 200)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# TODO: plot sigmoid, tanh_act, relu on axes[0]
# TODO: plot sigmoid_grad, tanh_grad, relu_grad on axes[1]

axes[0].set_title("Activation functions")
axes[0].legend()
axes[1].set_title("Derivatives")
axes[1].legend()
plt.tight_layout()
plt.show()

### Questions
1. For which range of inputs does the sigmoid gradient vanish? Why is this a problem for deep networks?
2. Why is ReLU preferred over sigmoid/tanh in most modern deep networks?
3. Why must we use softmax (not sigmoid) at the output of a multi-class classifier?

_Your answers here._

---
## Part 2 — Forward Pass in a Two-Layer Network

Architecture:
$$h = f(W_1 x + b_1), \quad \hat{y} = \text{softmax}(W_2 h + b_2)$$

In [ ]:
def init_params(input_dim: int, hidden_dim: int, output_dim: int, seed: int = 42) -> dict:
    """
    Initialize W1, b1, W2, b2.
    Weights ~ N(0, 0.01), biases = 0.
    """
    rng = np.random.default_rng(seed)
    # TODO: implement
    pass

# Test
params = init_params(input_dim=4, hidden_dim=8, output_dim=3)
print("W1 shape:", params["W1"].shape)
print("W2 shape:", params["W2"].shape)

In [ ]:
def forward(x: np.ndarray, params: dict, activation) -> tuple[np.ndarray, dict]:
    """
    Forward pass.
    Returns (y_hat, cache) where cache contains all intermediate values
    needed for backpropagation: x, z1, h, z2, y_hat.
    """
    # TODO: implement
    # z1 = W1 @ x + b1
    # h  = activation(z1)
    # z2 = W2 @ h + b2
    # y_hat = softmax(z2)
    pass

# Test
x_test = np.random.randn(4)
y_hat, cache = forward(x_test, params, tanh_act)
print("y_hat:", y_hat, "sum:", y_hat.sum())

In [ ]:
def cross_entropy_loss(y_hat: np.ndarray, y_true: np.ndarray) -> float:
    """
    Cross-entropy loss.
    y_hat: predicted probabilities (shape C)
    y_true: one-hot target (shape C)
    Returns scalar loss.
    """
    # TODO: implement (clip y_hat for numerical stability)
    pass

# Test: perfect prediction should give near-zero loss
y_true = np.array([0.0, 1.0, 0.0])
y_perfect = np.array([1e-9, 1.0 - 2e-9, 1e-9])
print("loss (perfect):", cross_entropy_loss(y_perfect, y_true))
print("loss (uniform):", cross_entropy_loss(np.array([1/3, 1/3, 1/3]), y_true))

### Questions
1. What does the cache returned by `forward` need to contain for backpropagation?
2. Why is numerical stability important when computing log-softmax?

_Your answers here._

---
## Part 3 — Backpropagation

Key formulas:
$$
\delta_2 = \hat{y} - y, \quad
\nabla_{W_2} = \delta_2 h^\top, \quad
\delta_1 = (W_2^\top \delta_2) \odot f'(z_1), \quad
\nabla_{W_1} = \delta_1 x^\top
$$

In [ ]:
def backward(x: np.ndarray, y_true: np.ndarray, cache: dict,
             params: dict, activation_grad) -> dict:
    """
    Backpropagation.
    activation_grad: derivative function of the hidden activation (e.g. tanh_grad).
    Returns dict with keys: dW1, db1, dW2, db2.
    """
    # TODO: implement
    # delta2 = y_hat - y_true       (softmax + cross-entropy combined gradient)
    # dW2    = np.outer(delta2, h)
    # db2    = delta2
    # delta1 = (W2.T @ delta2) * activation_grad(z1)
    # dW1    = np.outer(delta1, x)
    # db1    = delta1
    pass

# Quick test: shapes should match params
y_true_test = np.array([0.0, 1.0, 0.0])
grads = backward(x_test, y_true_test, cache, params, tanh_grad)
for k, v in grads.items():
    print(k, v.shape, "matches params?", v.shape == params[k.replace("d", "", 1)].shape)

In [ ]:
def gradient_check(x: np.ndarray, y_true: np.ndarray, params: dict,
                   activation, activation_grad, eps: float = 1e-5) -> dict:
    """
    Numerical gradient check.
    Returns dict of relative errors per parameter.
    Relative error = |analytic - numeric| / (|analytic| + |numeric| + 1e-8)
    """
    # TODO: implement
    # For each parameter matrix / vector:
    #   for each element theta_i:
    #     perturb +eps -> compute loss_plus
    #     perturb -eps -> compute loss_minus
    #     numeric_grad = (loss_plus - loss_minus) / (2 * eps)
    # Compare with analytic gradient from backward()
    pass

rel_errors = gradient_check(x_test, y_true_test, params, tanh_act, tanh_grad)
for k, err in rel_errors.items():
    print(f"{k}: max relative error = {err:.2e}")

In [ ]:
def sgd_update(params: dict, grads: dict, lr: float) -> None:
    """
    In-place SGD update: theta -= lr * grad_theta.
    """
    # TODO: implement
    pass

### Questions
1. What is the "vanishing gradient" problem, and which activation functions are most prone to it?
2. Why does gradient checking use a two-sided finite difference rather than a one-sided one?

_Your answers here._

---
## Part 4 — Neural Language Model (Fixed-Window)

Architecture (Bengio et al., 2003):
$$x = [e(w_{t-n+1}); \dots; e(w_{t-1})], \quad h = \tanh(W_1 x + b_1), \quad P(\cdot) = \text{softmax}(W_2 h + b_2)$$

In [ ]:
# Corpus: a small set of sentences for demonstration
raw_sentences = [
    "i like nlp",
    "i like deep learning",
    "i like learning new things",
    "nlp is fun",
    "deep learning is fun",
    "learning is useful",
    "nlp is useful",
    "i enjoy nlp and deep learning",
    "deep learning models learn representations",
    "language models predict the next word",
]

BOS = "<s>"
EOS = "</s>"
UNK = "<UNK>"

sentences_tok = [nltk.word_tokenize(s.lower()) for s in raw_sentences]

# Build vocabulary
word_freq = Counter(w for s in sentences_tok for w in s)
vocab = [BOS, EOS, UNK] + sorted(w for w, c in word_freq.items() if c >= 1)
word_to_id = {w: i for i, w in enumerate(vocab)}
id_to_word = {i: w for w, i in word_to_id.items()}
V = len(vocab)
print(f"Vocabulary size: {V}")
print("Vocab sample:", vocab[:10])

In [ ]:
def lookup_embeddings(context_ids: list[int], E: np.ndarray) -> np.ndarray:
    """
    Retrieve and concatenate embedding vectors for context_ids.
    E: (V, d) embedding matrix.
    Returns: 1-D array of shape ((n-1)*d,).
    """
    # TODO: implement
    pass

# Test
d = 8  # embedding dimension
E_test = np.random.randn(V, d) * 0.01
ctx = [word_to_id[BOS], word_to_id.get("i", word_to_id[UNK])]
emb = lookup_embeddings(ctx, E_test)
print("Embedding shape:", emb.shape, "expected:", (len(ctx) * d,))

In [ ]:
def nlm_forward(context_ids: list[int], E: np.ndarray, params: dict) -> tuple:
    """
    NLM forward pass.
    Returns (y_hat, cache, x) where x is the concatenated embedding input.
    """
    # TODO: implement
    # 1. x = lookup_embeddings(context_ids, E)
    # 2. y_hat, cache = forward(x, params, tanh_act)
    # 3. return y_hat, cache, x
    pass

# Test
n = 2  # bigram context (1 previous word)
H = 16
nlm_params = init_params(input_dim=(n - 1) * d, hidden_dim=H, output_dim=V)
y_hat_test, _, _ = nlm_forward(ctx, E_test, nlm_params)
print("y_hat shape:", y_hat_test.shape, "sum:", y_hat_test.sum().round(6))

In [ ]:
def nlm_backward(context_ids: list[int], y_true: np.ndarray,
                 y_hat: np.ndarray, cache: dict, x: np.ndarray,
                 E: np.ndarray, params: dict) -> tuple[dict, dict]:
    """
    NLM backward pass.
    Returns:
      param_grads: dict with dW1, db1, dW2, db2
      embed_grads: dict mapping word_id -> gradient vector (shape d,)
                   (accumulate gradients if a word appears multiple times)
    """
    # TODO: implement
    # 1. param_grads = backward(x, y_true, cache, params, tanh_grad)
    # 2. dx = W1.T @ delta1  (gradient w.r.t. concatenated embedding x)
    # 3. split dx into d-dimensional slices, one per context word
    # 4. accumulate into embed_grads[word_id]
    pass

In [ ]:
def build_ngram_examples(sentences_tokens: list[list[str]], n: int,
                          word_to_id: dict[str, int]) -> list[tuple]:
    """
    Build (context_ids, target_id) training pairs.
    Prepend (n-1) BOS tokens to each sentence.
    Map unknown words to UNK.
    Returns list of (context_ids: list[int], target_id: int).
    """
    # TODO: implement
    pass

examples = build_ngram_examples(sentences_tok, n=2, word_to_id=word_to_id)
print(f"Total training examples (n=2): {len(examples)}")
print("First 3 examples (context_ids, target_id):", examples[:3])

In [ ]:
def train_nlm(
    sentences_tokens: list[list[str]],
    word_to_id: dict[str, int],
    n: int = 2,
    d: int = 16,
    H: int = 32,
    lr: float = 0.05,
    epochs: int = 20,
    seed: int = 42,
) -> tuple:
    """
    Train a fixed-window neural language model.
    Returns (E, params, loss_history).
    """
    V = len(word_to_id)
    rng = np.random.default_rng(seed)

    # Initialize embeddings and network params
    # TODO: E = rng.normal(0, 0.01, (V, d))
    # TODO: params = init_params((n - 1) * d, H, V, seed=seed)

    examples = build_ngram_examples(sentences_tokens, n, word_to_id)
    loss_history = []

    for epoch in range(1, epochs + 1):
        rng.shuffle(examples)  # shuffle in-place copy
        total_loss = 0.0

        for context_ids, target_id in examples:
            # One-hot target
            y_true = np.zeros(V)
            y_true[target_id] = 1.0

            # TODO: forward pass
            # TODO: compute loss, accumulate total_loss
            # TODO: backward pass
            # TODO: sgd_update params
            # TODO: update embeddings: E[word_id] -= lr * embed_grads[word_id]
            pass

        avg_loss = total_loss / max(len(examples), 1)
        loss_history.append(avg_loss)
        if epoch % 5 == 0 or epoch == 1:
            print(f"Epoch {epoch:3d} | avg loss = {avg_loss:.4f}")

    return E, params, loss_history

E, nlm_params, loss_history = train_nlm(
    sentences_tok, word_to_id, n=2, d=16, H=32, lr=0.05, epochs=30
)

In [ ]:
# Plot training loss
plt.plot(loss_history)
plt.xlabel("Epoch")
plt.ylabel("Average cross-entropy loss")
plt.title("NLM training loss")
plt.show()

### Questions
1. How does the neural LM handle unseen words at test time differently from an n-gram model with add-one smoothing?
2. What does the embedding matrix $E$ represent, and how does it differ from a one-hot input representation?
3. Why is the context window size $n$ a hyperparameter in the neural LM, just as in n-gram models?

_Your answers here._

---
## Part 5 — Perplexity and Comparison with N-gram Models

In [ ]:
def nlm_perplexity(
    sentences_tokens: list[list[str]],
    E: np.ndarray,
    params: dict,
    n: int,
    word_to_id: dict[str, int],
) -> float:
    """
    Compute perplexity of the NLM on a list of tokenized sentences.
    PP = exp(- (1/N) * sum log P(target | context))
    """
    # TODO: implement
    # Use build_ngram_examples to get (context_ids, target_id) pairs
    # For each pair, run nlm_forward and extract log prob of target
    pass

train_pp = nlm_perplexity(sentences_tok, E, nlm_params, n=2, word_to_id=word_to_id)
print(f"NLM train perplexity (n=2): {train_pp:.2f}")

In [ ]:
# --- Bigram add-one baseline (from Week 04 / included here for self-containment) ---

def build_bigram_lm(sentences_tokens, vocab):
    V = len(vocab)
    unigram = Counter()
    bigram = Counter()
    for sent in sentences_tokens:
        padded = [BOS] + sent + [EOS]
        for w in padded:
            unigram[w] += 1
        for w_prev, w in zip(padded, padded[1:]):
            bigram[(w_prev, w)] += 1

    def p_add1(w_prev, w):
        w_prev = w_prev if w_prev in vocab else UNK
        w = w if w in vocab else UNK
        return (bigram[(w_prev, w)] + 1) / (unigram[w_prev] + V)

    return p_add1


def ngram_perplexity(sentences_tokens, p_bigram):
    total_log = 0.0
    total_n = 0
    for sent in sentences_tokens:
        padded = [BOS] + sent + [EOS]
        for w_prev, w in zip(padded, padded[1:]):
            total_log += math.log(p_bigram(w_prev, w))
            total_n += 1
    return math.exp(-total_log / max(total_n, 1))


bigram_vocab = set(vocab)
p_add1 = build_bigram_lm(sentences_tok, bigram_vocab)
bigram_pp = ngram_perplexity(sentences_tok, p_add1)
print(f"Bigram add-one train perplexity: {bigram_pp:.2f}")

In [ ]:
# Coding Task B — Fill in the comparison table (train and a held-out split)
# TODO: split sentences_tok into train/test (e.g. 8/2 sentences)
# TODO: train bigram LM on train, evaluate on test
# TODO: train NLM (n=2 and n=3) on train, evaluate on test
# TODO: print results as a table

print("| Model              | Train PP | Test PP |")
print("|-------------------|----------|---------|")
# print each row here

In [ ]:
# Coding Task C — Nearest-neighbor words by learned embeddings

def cosine(u: np.ndarray, v: np.ndarray) -> float:
    norm = np.linalg.norm(u) * np.linalg.norm(v)
    return float(np.dot(u, v) / norm) if norm > 0 else 0.0

def nearest_neighbors(query_word: str, E: np.ndarray, word_to_id: dict,
                      id_to_word: dict, top_k: int = 5) -> list:
    """
    Return top_k nearest neighbors by cosine similarity over rows of E.
    """
    # TODO: implement
    pass

for qw in ["nlp", "learning", "fun"]:
    neighbors = nearest_neighbors(qw, E, word_to_id, id_to_word, top_k=5)
    print(f"Nearest to '{qw}':", neighbors)

### Questions
1. Does the neural LM achieve lower test perplexity than add-one bigram? Under what conditions might it not?
2. How do the embedding-based nearest neighbors differ from the co-occurrence-based ones from Week 04?
3. What are two limitations of the fixed-window neural LM compared to recurrent or transformer models?

_Your answers here._

---
## Part 6 — Mini XOR Network

XOR cannot be learned by a linear model. Show that a network with one hidden layer can.

In [ ]:
# XOR dataset (2-class, one-hot targets)
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
Y_xor_labels = [0, 1, 1, 0]  # 0=False, 1=True
Y_xor = np.eye(2)[Y_xor_labels]  # one-hot

# TODO: initialize a network with input_dim=2, hidden_dim=4, output_dim=2
# TODO: train for up to 5000 steps with lr=0.1, using relu or tanh activation
# TODO: print final predictions and loss

# Expected output: inputs [0,1] and [1,0] should be classified as class 1,
# and [0,0] and [1,1] should be classified as class 0.

### Question
1. Does a single linear layer (no hidden activation) converge on XOR? Why or why not?

_Your answer here._

---
## Submission
- Ensure all `TODO` sections are completed.
- Answer all written questions in the markdown cells above.
- Submit your completed `.ipynb` notebook as instructed.